# SVM MODEL Experiments

**Objective:**

This notebook evaluates the performance of the SVM model using different strategies to handle class imbalance:

- Baseline
- Class-weight
- SMOTE
- SMOTEENN

**All experiments use:**
- A shared preprocessing pipeline
- Stratified K-Fold cross-validation
- A unified evaluation framework

Results are stored in a shared results table for comparison with other models.

## Setup

In [6]:
# Add project root to system path so we can import from src/
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent / "bank-deposit-prediction"
sys.path.append(str(PROJECT_ROOT))

# Standard libraries
import pandas as pd

# Modeling
from sklearn.pipeline import Pipeline
# For the standard Support Vector Classifier (what you are likely using)
from sklearn.svm import SVC

# Imbalanced learning tools
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

# Shared project modules
from src.shared import get_cv, get_preprocessing_steps
from src.evaluation import evaluate_model, save_results

## Data Loading

In [7]:
data_path = PROJECT_ROOT / "data" / "raw" / "bank2.csv"

df = pd.read_csv(data_path, sep=';')

y = df['y'].map({'yes': 1, 'no': 0})
X = df.drop(columns=['y'])

## SVM PIPELINE SETUP 

In [8]:
cv = get_cv()

## Baseline SVM Model

In [9]:
# Store experiment results
all_results = []

# Baseline Random Forest (no imbalance handling)
baseline_svm_pipeline = Pipeline(
    get_preprocessing_steps() + [
        ('model', SVC(kernel='rbf', probability=True, random_state=42, class_weight=None))
    ]
)

# Evaluate model using cross-validation
result_svm_baseline = evaluate_model(
    "SVM",
    "Baseline",
    baseline_svm_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_svm_baseline)
result_svm_baseline


c:\Users\marwa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marwa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\marwa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

{'Model': 'SVM',
 'Strategy': 'Baseline',
 'Accuracy': '0.8830 ± 0.0022',
 'Precision': '0.0000 ± 0.0000',
 'Recall': '0.0000 ± 0.0000',
 'F1': '0.0000 ± 0.0000',
 'PR-AUC': '0.2661 ± 0.0278',
 'ROC-AUC': '0.6339 ± 0.0320'}

## Class Weight (Imbalance Handling)

In [10]:
# Class weight SVM
cw_rf_pipeline = Pipeline(
    get_preprocessing_steps() + [
        ('model', SVC(kernel='rbf', class_weight='balanced',probability=True, random_state=42))
    ]
)

# Evaluate model using cross-validation
result_svm_cw = evaluate_model(
    "SVM",
    "ClassWeight",
    cw_rf_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_svm_cw)

result_svm_cw


{'Model': 'SVM',
 'Strategy': 'ClassWeight',
 'Accuracy': '0.8255 ± 0.0150',
 'Precision': '0.2698 ± 0.0278',
 'Recall': '0.2938 ± 0.0249',
 'F1': '0.2798 ± 0.0182',
 'PR-AUC': '0.2834 ± 0.0298',
 'ROC-AUC': '0.7042 ± 0.0218'}

In [11]:
# SMOTE SVM
smote_svm_pipeline = ImbPipeline(
    get_preprocessing_steps() + [
        ('smote', SMOTE(random_state=42)),
        ('model', SVC(kernel='rbf',probability=True, random_state=42))
    ]
)

# Evaluate model
result_svm_smote = evaluate_model(
    "SVM",
    "SMOTE",
    smote_svm_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_svm_smote)

result_svm_smote

{'Model': 'SVM',
 'Strategy': 'SMOTE',
 'Accuracy': '0.8182 ± 0.0254',
 'Precision': '0.2639 ± 0.0424',
 'Recall': '0.3033 ± 0.0240',
 'F1': '0.2797 ± 0.0255',
 'PR-AUC': '0.2801 ± 0.0343',
 'ROC-AUC': '0.7021 ± 0.0230'}

## SMOTEEEN 

In [12]:
# SMOTEENN Random Forest
smoteenn_svm_pipeline = ImbPipeline(
    get_preprocessing_steps() + [
        ('smoteenn', SMOTEENN(random_state=42)),
        ('model', SVC(kernel='rbf', probability=True, random_state=42))
    ]
)

# Evaluate model
result_svm_smoteenn = evaluate_model(
    "SVM",
    "SMOTEENN",
    smoteenn_svm_pipeline,
    X,
    y,
    cv
)

# Save and display results
all_results.append(result_svm_smoteenn)

result_svm_smoteenn


{'Model': 'SVM',
 'Strategy': 'SMOTEENN',
 'Accuracy': '0.1157 ± 0.0009',
 'Precision': '0.1153 ± 0.0004',
 'Recall': '1.0000 ± 0.0000',
 'F1': '0.2067 ± 0.0006',
 'PR-AUC': '0.2297 ± 0.0277',
 'ROC-AUC': '0.6819 ± 0.0204'}

## Results

In [13]:
# Save results
results_df = pd.DataFrame(all_results)
save_results(results_df)

# Read experiment_results table
results_path = PROJECT_ROOT / "results" / "experiment_results.csv"
results = pd.read_csv(results_path)
results

,Model,Strategy,Accuracy,Precision,Recall,F1,PR-AUC,ROC-AUC
0,XGBoost,Baseline,0.8830 ± 0.0048,0.4765 ± 0.0576,0.1784 ± 0.0353,0.2590 ± 0.0447,0.3037 ± 0.0329,0.7020 ± 0.0049
1,XGBoost,ClassWeight,0.8498 ± 0.0070,0.3300 ± 0.0308,0.2936 ± 0.0394,0.3099 ± 0.0313,0.2791 ± 0.0381,0.6806 ± 0.0194
2,XGBoost,SMOTE,0.8834 ± 0.0023,0.4829 ± 0.0311,0.1765 ± 0.0372,0.2567 ± 0.0431,0.3178 ± 0.0177,0.7172 ± 0.0150
3,XGBoost,SMOTEENN,0.8290 ± 0.0041,0.3266 ± 0.0061,0.4549 ± 0.0160,0.3800 ± 0.0061,0.3323 ± 0.0215,0.7270 ± 0.0142
4,RandomForest,Baseline,0.8885 ± 0.0028,0.5940 ± 0.0983,0.1209 ± 0.0231,0.1989 ± 0.0308,0.3482 ± 0.0273,0.7333 ± 0.0184
5,RandomForest,ClassWeight,0.8910 ± 0.0049,0.6479 ± 0.1295,0.1151 ± 0.0303,0.1947 ± 0.0493,0.3418 ± 0.0378,0.7384 ± 0.0211
6,RandomForest,SMOTE,0.8848 ± 0.0051,0.5003 ± 0.0823,0.1516 ± 0.0294,0.2320 ± 0.0416,0.3234 ± 0.0372,0.7252 ± 0.0239
7,RandomForest,SMOTEENN,0.8275 ± 0.0083,0.3171 ± 0.0158,0.4281 ± 0.0186,0.3639 ± 0.0125,0.3064 ± 0.0331,0.7207 ± 0.0273
8,LogisticRegression,Baseline,0.8923 ± 0.0037,0.6605 ± 0.1145,0.1439 ± 0.0290,0.2342 ± 0.0399,0.3482 ± 0.0375,0.7247 ± 0.0143
9,LogisticRegression,ClassWeight,0.7125 ± 0.0128,0.2188 ± 0.0073,0.5815 ± 0.0427,0.3176 ± 0.0122,0.3407 ± 0.0344,0.7238 ± 0.0130
